# A/B Testing: Does a New Button Increase Sign-ups?

## Why Do We Need a Statistical Test?

Imagine you run an A/B test comparing two signup buttons:

| Variant | Conversion Rate |
|---------|-----------------|
| **A (Control)** | 10% |
| **B (Treatment)** | 12% |

At first glance, **B looks better** than A.

But here's the key question: **Is this 2 percentage-point difference a real treatment effect, or could it simply be due to random chance?**

This tutorial teaches you to answer this question using hypothesis testing and p-values.

## Setup: Import the Tools We'll Use

In [20]:
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

---

# PHASE 1: PLAN

## What are we testing?

We want to know whether a redesigned signup button (Variant B) **truly increases signups** compared to the current button (Variant A), or if the observed 10% → 12% difference is just random variation.

**Our starting assumption (Null Hypothesis):**
> The new button has **no real effect**. Any difference we observe is just random chance.

**Our alternative (Alternative Hypothesis):**
> The new button **does change** the signup rate.

In mathematical notation:
- **H₀:** p_A = p_B (no difference)
- **H₁:** p_A ≠ p_B (there is a difference)

**Key insight:** We always *start by assuming H₀ is true*. The job of the statistical test is to see if our observed data provides strong enough evidence to reject this assumption.

---

# PHASE 2: RUN

## The Experiment Results

We ran the experiment fairly and randomly assigned visitors to each variant:

| Metric | Variant A (Control) | Variant B (Treatment) |
|--------|------------|-----------|
| Visitors tested | 5,000 | 5,000 |
| Signups | 500 | 600 |
| **Conversion rate** | **10%** | **12%** |
| **Observed difference** | — | **+2 percentage points** |

The data collection is complete. We observed that Variant B had a 2 percentage-point higher signup rate.

**But does this difference mean Variant B is truly better, or is it just random variation?**

That's what we'll test next.

In [21]:
# Experiment data
n_a = 5000  # Sample size for Variant A
n_b = 5000  # Sample size for Variant B
signups_a = 500  # Number of signups from Variant A
signups_b = 600  # Number of signups from Variant B

# Calculate conversion rates
rate_a = signups_a / n_a
rate_b = signups_b / n_b
observed_diff = rate_b - rate_a

print("=== OBSERVED RESULTS ===")
print(f"Variant A conversion rate: {rate_a:.1%}")
print(f"Variant B conversion rate: {rate_b:.1%}")
print(f"Observed difference: {observed_diff:.1%}")
print(f"\nVariant B is {observed_diff:.1%} higher than Variant A")

=== OBSERVED RESULTS ===
Variant A conversion rate: 10.0%
Variant B conversion rate: 12.0%
Observed difference: 2.0%

Variant B is 2.0% higher than Variant A


---

# PHASE 3: ANALYZE

## Understanding Random Variation

**Key idea:** Even if both variants had the *exact same* true conversion rate, random sampling variation means we wouldn't get exactly 10% and 10%. We'd see small differences just by chance.

The question becomes: **Is the observed 2 percentage-point difference surprising if both variants are truly equal?**

This is exactly what a p-value tells us.

## Step 1: Choose a Confidence Threshold

Before we do the test, we decide: **How much evidence do we need to reject our null hypothesis?**

The standard choice is:
$$\alpha = 0.05$$

**What this means:**
- We're willing to accept a **5% false positive rate**
- In other words, if we repeated this experiment 100 times with variants that are truly identical, we'd wrongly conclude one was better in about 5 of those experiments

**Important:** This does NOT mean "there's a 5% chance the null hypothesis is true." It means "we set our threshold so we'd have about 5% false positives."

For now, let's proceed with α = 0.05.

In [23]:
# Set the significance level
alpha = 0.05
print(f"Significance level (alpha): {alpha}")
print(f"This means we reject H0 only if the p-value is less than {alpha}")

Significance level (alpha): 0.05
This means we reject H0 only if the p-value is less than 0.05


## Step 2: Run the Statistical Test

For comparing two conversion rates, we use a **two-proportion z-test**. This test asks:

> **Assuming both variants truly have the same conversion rate, how extreme is the observed difference we see?**

We calculate:
1. **Pooled conversion rate** – the overall rate combining both groups
2. **Standard error** – how much variation we expect due to random sampling
3. **Z-score** – how many standard errors away our observed difference is
4. **P-value** – the probability of seeing this extreme a result if H₀ is true

In [24]:
# Step 1: Calculate the pooled conversion rate
# (what conversion rate would we expect if both variants are truly identical?)
pooled_rate = (signups_a + signups_b) / (n_a + n_b)

print(f"Pooled conversion rate: {pooled_rate:.4f} ({pooled_rate:.2%})")

# Step 2: Calculate the standard error
# This tells us: how much would we expect conversion rates to vary by chance?
se = np.sqrt(pooled_rate * (1 - pooled_rate) * (1/n_a + 1/n_b))

print(f"Standard error: {se:.6f}")

# Step 3: Calculate the Z-score
# This tells us: how many standard errors away is our observed difference?
z_score = observed_diff / se

print(f"\n=== Z-TEST RESULT ===")
print(f"Z-score: {z_score:.4f}")
print(f"\nThis means the observed difference is {z_score:.2f} standard errors away from zero")

Pooled conversion rate: 0.1100 (11.00%)
Standard error: 0.006258

=== Z-TEST RESULT ===
Z-score: 3.1960

This means the observed difference is 3.20 standard errors away from zero


## Step 3: Calculate the P-value

**Definition of p-value:**
> Assuming the null hypothesis is true, the p-value is the probability of observing a result *at least as extreme* as what we actually observed.

In plain English:
> **If both buttons were truly identical, how likely would we be to see a 2 percentage-point difference (or even larger) just by random chance?**

For a two-sided test, we calculate the probability in both tails of the distribution.

In [26]:
# Calculate the p-value for a two-sided test
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

print(f"P-value: {p_value:.6f}")
print(f"P-value as percentage: {p_value:.2%}")
print(f"\n=== INTERPRETATION ===")
print(f"If both variants truly had the same conversion rate,")
print(f"we would observe a difference this extreme or greater")
print(f"{p_value:.2%} of the time, just by random chance.")

P-value: 0.001393
P-value as percentage: 0.14%

=== INTERPRETATION ===
If both variants truly had the same conversion rate,
we would observe a difference this extreme or greater
0.14% of the time, just by random chance.


## Step 4: Compare P-value to Significance Threshold

Now we compare our p-value to α = 0.05:

In [27]:
print(f"P-value: {p_value:.6f}")
print(f"Significance level (alpha): {alpha}")
print()

if p_value < alpha:
    print(f"Check: P-value ({p_value:.6f}) < alpha ({alpha})")
    print("\n=> RESULT: STATISTICALLY SIGNIFICANT")
    print("\nWe reject the null hypothesis H0.")
    print("There is evidence that Variant B truly changes the conversion rate.")
else:
    print(f"Check: P-value ({p_value:.6f}) >= alpha ({alpha})")
    print("\n=> RESULT: NOT STATISTICALLY SIGNIFICANT")
    print("\nWe fail to reject the null hypothesis H0.")
    print("We do not have enough evidence to conclude that Variant B")
    print("produces a real difference from Variant A.")

P-value: 0.001393
Significance level (alpha): 0.05

Check: P-value (0.001393) < alpha (0.05)

=> RESULT: STATISTICALLY SIGNIFICANT

We reject the null hypothesis H0.
There is evidence that Variant B truly changes the conversion rate.


### What the P-value Does NOT Mean

This is critically important. A p-value is **frequently misinterpreted**:

**WRONG:** "The p-value is the probability that the null hypothesis is true"
- **CORRECT:** The p-value is the probability of observing this extreme a result *assuming* H₀ is true

**WRONG:** "A p-value of 0.03 means Variant B has a 97% chance of being better"
- **CORRECT:** It means if H₀ were true, we'd see this extreme a result 3% of the time

**WRONG:** "If p > 0.05, then H₀ is proven true"
- **CORRECT:** If p > 0.05, we simply don't have enough evidence to reject H₀. That's different from proving it's true.

**The key insight:** The p-value is a measure of *how surprising your result is*, assuming H₀ is true. That's it.

## Confidence Interval: Another Way to See the Result

A **95% confidence interval** gives us a range of plausible values for the true difference between the two conversion rates:

In [28]:
# Calculate 95% confidence interval for the difference
# 95% CI = observed_diff +/- 1.96 * SE

margin_of_error = 1.96 * se
ci_lower = observed_diff - margin_of_error
ci_upper = observed_diff + margin_of_error

print(f"Observed difference: {observed_diff:.4f} ({observed_diff:.2%})")
print(f"Margin of error: {margin_of_error:.4f} ({margin_of_error:.2%})")
print(f"\n95% Confidence Interval: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"In percentage points: [{ci_lower:.2%}, {ci_upper:.2%}]")
print()
print("Interpretation:")
print(f"We are 95% confident that the true difference between")
print(f"Variant A and Variant B lies somewhere in this range.")
print()

if ci_lower > 0 and ci_upper > 0:
    print("The confidence interval does NOT include zero.")
    print("  This matches our p-value result: statistically significant")
elif ci_lower < 0 and ci_upper < 0:
    print("The confidence interval does NOT include zero.")
    print("  This matches our p-value result: statistically significant")
else:
    print("The confidence interval INCLUDES zero.")
    print("  This matches our p-value result: not statistically significant")

Observed difference: 0.0200 (2.00%)
Margin of error: 0.0123 (1.23%)

95% Confidence Interval: [0.0077, 0.0323]
In percentage points: [0.77%, 3.23%]

Interpretation:
We are 95% confident that the true difference between
Variant A and Variant B lies somewhere in this range.

The confidence interval does NOT include zero.
  This matches our p-value result: statistically significant


---

# PHASE 4: DECIDE

## Statistical Significance ≠ Practical Significance

**Here's something crucial:** A result can be statistically significant but NOT practically meaningful for your business.

**Example:**
Imagine a much larger experiment gives:
- Variant A: 10.00%
- Variant B: 10.05%
- p-value: 0.001 (highly statistically significant!)

The difference is real from a statistical perspective, BUT a 0.05 percentage-point improvement might be too small to justify the costs of implementation.

**Therefore, the final business decision requires THREE things:**

1. **Statistical Evidence** – Is the p-value < 0.05?
2. **Effect Size** – How big is the difference? (2 percentage points vs. 0.05 percentage points)
3. **Business Value** – Is the improvement worth implementing?

In [29]:
print("=== DECISION FRAMEWORK ===")
print()
print(f"1. STATISTICAL SIGNIFICANCE")
print(f"   P-value: {p_value:.6f}")
if p_value < alpha:
    print(f"   ✓ Statistically significant (p < {alpha})")
    is_sig = True
else:
    print(f"   ✗ Not statistically significant (p >= {alpha})")
    is_sig = False
print()
print(f"2. EFFECT SIZE")
print(f"   Observed difference: {observed_diff:.2%}")
print(f"   Relative improvement: {(observed_diff/rate_a):.1%} higher than Variant A")
print()
print(f"3. BUSINESS VALUE")
print(f"   Would a {observed_diff:.2%} improvement justify implementation?")
print(f"   (This depends on your business goals and costs)")
print()
if is_sig:
    print("RECOMMENDATION:")
    print("The difference IS statistically significant. The next question:")
    print("Is the effect size large enough to matter for your business?")
else:
    print("RECOMMENDATION:")
    print("The difference is NOT statistically significant.")
    print("Treat A and B as performing equally. Keep using Variant A")
    print("or test other changes.")

=== DECISION FRAMEWORK ===

1. STATISTICAL SIGNIFICANCE
   P-value: 0.001393
   ✓ Statistically significant (p < 0.05)

2. EFFECT SIZE
   Observed difference: 2.00%
   Relative improvement: 20.0% higher than Variant A

3. BUSINESS VALUE
   Would a 2.00% improvement justify implementation?
   (This depends on your business goals and costs)

RECOMMENDATION:
The difference IS statistically significant. The next question:
Is the effect size large enough to matter for your business?


## Planning for the Future: Sample Size and Duration

Before running your next experiment, you should plan:
1. **How many visitors do you need to reliably detect a real difference?**
2. **How long will the experiment take?**

In [30]:
# SAMPLE SIZE CALCULATION
# How many visitors per variant do we need to reliably detect an effect?

# Our assumptions
p0 = 0.10  # Baseline conversion rate (Variant A)
delta = 0.02  # Minimum effect size we want to detect (2 percentage points)
alpha = 0.05  # Significance level
power = 0.80  # Statistical power (1 - beta)

# From statistics tables
z_alpha = stats.norm.ppf(1 - alpha/2)
z_beta = stats.norm.ppf(power)

# Effect size for two proportions
p1 = p0 + delta
effect_size = (p1 - p0) / np.sqrt(p0 * (1 - p0))

# Sample size formula
n_needed = 2 * ((z_alpha + z_beta) / effect_size)**2

print("=== SAMPLE SIZE PLANNING ===")
print(f"Baseline conversion rate: {p0:.1%}")
print(f"Desired effect size: {delta:.1%}")
print(f"Significance level: {alpha}")
print(f"Statistical power: {power:.0%}")
print()
print(f"Sample size needed PER VARIANT: {int(np.ceil(n_needed)):,}")
print(f"Total visitors needed: {int(np.ceil(2 * n_needed)):,}")
print()
print("This means you need to test at least", int(np.ceil(n_needed)), "visitors per variant")
print("to reliably detect a 2 percentage-point improvement with 80% power.")

=== SAMPLE SIZE PLANNING ===
Baseline conversion rate: 10.0%
Desired effect size: 2.0%
Significance level: 0.05
Statistical power: 80%

Sample size needed PER VARIANT: 3,532
Total visitors needed: 7,064

This means you need to test at least 3532 visitors per variant
to reliably detect a 2 percentage-point improvement with 80% power.


In [31]:
# EXPERIMENT DURATION CALCULATION
# Given daily traffic, how long will the experiment take?

daily_traffic = 1000  # Visitors per day
traffic_split = 0.5  # Each variant gets 50% of traffic

days_needed = n_needed / (daily_traffic * traffic_split)

print("=== EXPERIMENT DURATION ===")
print(f"Daily visitors: {daily_traffic:,}")
print(f"Traffic split: {traffic_split:.0%} to each variant")
print()
print(f"Experiment duration: {days_needed:.1f} days ({days_needed/7:.1f} weeks)")
print()
print("Note: This assumes constant daily traffic and a 50-50 split between variants")

=== EXPERIMENT DURATION ===
Daily visitors: 1,000
Traffic split: 50% to each variant

Experiment duration: 7.1 days (1.0 weeks)

Note: This assumes constant daily traffic and a 50-50 split between variants


---

## Summary: The Complete A/B Testing Process

### **PLAN:** Define the Question
- State your hypothesis: "Does Variant B change the conversion rate?"
- Choose your significance level: α = 0.05

### **RUN:** Collect Fair Data
- Randomly assign visitors to variants
- Record conversion rates for each
- Our example: A = 10%, B = 12%

### **ANALYZE:** Test for Statistical Significance
- Calculate a test statistic (Z-score) measuring how extreme the difference is
- Compute the p-value: probability of this extreme result if H₀ is true
- Compare p-value to α:
  - If p < 0.05 → reject H₀ (statistically significant)
  - If p ≥ 0.05 → fail to reject H₀ (not statistically significant)

### **DECIDE:** Consider Business Value
- Is the result statistically significant?
- Is the effect size practically meaningful? (2% vs. 0.05%?)
- Does the improvement justify implementation?

---

## Key Takeaway for Students

> **The p-value helps us judge whether an observed A/B difference could reasonably be explained by random variation if there were actually no treatment effect.**

Or more simply:

> **B looks better. The statistical test tells us whether we have enough evidence to believe the difference is real rather than random variation.**